Memory in LangChain


Legacy (deprecated in 0.3.1) in favor of LCEL-native patterns
ConversationBufferMemory
ConversationSummaryMemory
ConversationBufferWindowMemory


Modern (recommended)
RunnableWithMessageHistory      Primary approach
Manual message management       Full control
LangGraph state                 For agents



#Modern message History Approach

Session store -> RunnableWithMessageHistory -> passed to the Chain


Each session store has an ID attached to it. Each session ID gets its own conversation history.
RunnableWithMessageHistory keeps record of  input_messages_key and history_messages_key

Memory Strategies Comparison

Strategy                              Tokens Used                           Best For

Full Buffer                           Grows linearly                      Short conversations
Window (k = 5)                         Fixed                              Long conversations
Summary                                Grows slowly                       Very long conversations
Summary + Buffer                       Moderate                           Balance of detail & context

Trimming Messages

trim_ messages

max_tokens=1000
strategy="last"
include_system=True

Persistent Memory with SQLite

In-Memory Storage
Messages in RAM
Lost on restart
Restart → All history gone

SQLite Storage
chat_history.db Persists to disk
Restart
History preserved


Usage Pattern
session_id -> SQLChatMessageHistory -> sqlite:///chat.db

In [ ]:
"""Conversation Memory in LangChain
Modern approaches to maintaining conversation context"""

from langchain_core import messages
from langchain_core.outputs import llm_result
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder #define the structure of the messages you want to send to an LLM.MessagesPlaceholder is used when you want to insert an existing list of chat messages into your prompt.
from langchain_core.messages import (
HumanMessage,
AIMessage,
SystemMessage,
trim_messages,
)
from langchain_core.chat_history import (
InMemoryChatMessageHistory, #is a LangChain class that basically gives you a place to store messages in Python memory.
BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from typing import Dict
from dotenv import load_dotenv

load_dotenv()

# llm = ChatOpenAI(model="gpt-40-mini", temperature=0.7)
llm=init_chat_model("gpt-40-mini")


def demo_basic_memory():

    """Basic conversation memory with RunnableWithMessageHistory."""
    print("=" * 60)
    print("BASIC CONVERSATION MEMORY")
    print("Using RunnableWithMessageHistory (modern approach)")
    print("=" * 60)
    
   
    # Prompt with history placeholder

    prompt = ChatPromptTemplate.from_messages ([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder(variable_name="history"),
     ("human", "{input}"),
    ])

    chain = prompt | llm | StrOutputParser()

    # creating a session storage dictionary for chat histories
    #Key is a string (session IDs) and value is a chat history object.
    store: Dict [str, InMemoryChatMessageHistory] = {}

    def get_session_history(session_id: str)-> BaseChatMessageHistory:
        if session_id not in store:
            store [session_id] = InMemoryChatMessageHistory()
        return store [session_id]
    
    #Whenever someone calls this chain, find their history using get_session_history(), put that history into the history part of my prompt, run the chain, and maintain the conversation history."You didn't manually append them. The wrapper did it.
    # wrapper around your chain that automatically loads and saves chat history for each session.
    chain_with_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
    )
    #in your RunnableWithMessageHistory setup, the .messages list is maintained automatically. You don't manually append to it.

    # Configuration for this session(manual but in real it has to be a dynamic creation)
    config = {"configurable": {"session_id": "user_123"}}

    # Conversation
    messages = [
    "Hi! My name is Anushka.",
    "I'm learning about LangChain.",
    "What's my name and what am I learning?",
    ]
    
    print("\nConversation:")

    for msg in messages:
        print(f"\nUser: {msg}")
        response=chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}")

    # Show stored history
    print(f"\n--- Stored History ({len(store ['user_123'].messages)} messages")

    for msg in store ["user_123"].messages:
        role = "Human" if isinstance (msg, HumanMessage) else "AI"
        print(f" {role}: {msg.content[:50]}...") 

#MESSAGE TRIMMING (to fit the particular context window)
def demo_message_trimming():
    """Trim messages to fit context window."""
    print("=" * 60)
    print("MESSAGE TRIMMING")
    print("Keep conversation within token limits")
    print("=" * 60)

    # Simulate a long conversation
    messages = [
    SystemMessage(content="You are a helpful coding assistant."),
    HumanMessage(content="What is Python?"),
    AIMessage(content="Python is a high-level programming language known for its use in AI"),
    HumanMessage(content="How do I install it?"),
    AIMessage(content="You can install Python from python.org or use package manager"),
    HumanMessage(content="What about pip?"),
    AIMessage(content="Pip is Python's package installer. It comes with Python"),
    HumanMessage(content="Can you summarize everything we discussed?"),
    ]

    print(f"\nOriginal:{len(messages)} messages")

    #Trim to last N tokens
    trimmed = trim_messages (
    messages,
    max_tokens=200,
    strategy="last", #Start from the most recent messages and keep going backward until the token limit is reached.
    token_counter=llm, #Different models tokenize text differently.Use this LLM's tokenizer to estimate/count how many tokens these messages consume.
    include_system=True, #Always keep the system message when trimming.
    allow_partial=False,#This controls whether LangChain is allowed to cut a message in half to make it fit the token limit.Don't include a partial message. Either keep the whole message or remove it."
    )

    print(f"After trimming (max 200 tokens): {len(trimmed)} messages")
    print("\nTrimmed messages:")
    for msg in trimmed:
        role = type(msg).__name__replace("Message", "")
        print(f" {role}: {msg.content[:60]}...")


# WINDOWED MEMORY (every message we send to LLM will cost us tokens, now if there are multiple exchanges, we are passing previous exchanges plus current and all should be within the context window, also as the no. of exchanges keep progressing the cost also keeps increasing until you hit the limit of context window.)
#Sliding window says to Only keep the last K exchanges,all the rest gets dropped.

def demo_windowed_memory():
    """Implement sliding window memory manually."""
    print("=" * 60)
    print("WINDOWED MEMORY (Keep Last K)")
    print("Fixed-size conversation window")
    print("=" * 60)

    #If the conversation becomes very long, sending all previous messages to the LLM becomes expensive.
    #sliding-window conversation memory Only remember the last K conversation pairs
    #The parent class InMemoryChatMessageHistory already knows how to store,add,retrieve messages, we are just inheriting from it.
    class WindowedChatHistory(InMemoryChatMessageHistory):
        """Chat history that keeps only last k message pairs."""
        def _init__(self, k: int = 3):
            super()._init__()
            self.k = k
        def add_messages (self, messages):
            super().add_messages (messages)
            #Keep only last k pairs (2k messages: human + ai)
            if len(self.messages) > self.k * 2:
                self.messages = self.messages [-(self.k * 2):]
    
    store:Dict[str,WindowedChatHistory]={} #A dictionary whose keys are strings and whose values are WindowedChatHistory objects.j

    #sessionId allows different conversations to have different histories.

    def get_session_history(session_id: str)->BaseChatMessageHistory:
        if session_id not in store:
           store [session_id] = WindowedChatHistory(k=2)
        return store [session_id]

    prompt=ChatPromptTemplate.from_messages ([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
    ])

    chain = prompt | llm | StrOutputParser()
    # RunnableWithMessageHistory is a ready-made class that adds conversation memory to another chain.
    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history, #It's a function that receives a session ID and returns the message history for that session.
        input_messages_key="input", #Which key in my input dictionary contains the current user input
        history_messages_key="history", #Where exactly should I put the conversation history in the input sent to my chain?
    )

# Take my chain.
# Use get_session_history() to find the conversation for the current session.
# The current user message is inside the "input" field.
# Put the previous messages inside the "history" field

    config={"configuration":{"session_id":"windowed_test"}}

    # Simulate a conversation with more than 2 pairs
    exchanges =[
        "My name is Paulo",
        "I live in Seattle",
        "I work as an AI engineer",
        "I have 2 cats",
        "What do you remember about me?",
        ]

    print("\nConversation with k = 2 window:")

    for i, msg in enumerate(exchanges, 1):
        print(f"\nUser: {msg}")
        response = chain_with_history.invoke({"input": msg}, config=config)
        print(f"AI: {response}")

        # Show window state after each exchange so we can SEE it sliding
        history = store["windowed_test"].messages
        print(f" [Window: {len (history)} msgs] ", end="")
        facts_in_memory = [ m.content[:40] for m in history if isinstance(m, HumanMessage)]

        print(f"Remembers: {facts_in_memory}")

    # Final state shows what survived and what was lost
    print("\n" + "=" * 60)
    print("RESULT: Window only kept last 2 exchanges!")
    print("Lost: name (Paulo), city (Seattle), AND job (AI engineer)")
    print("Kept: cats + the 'remember' question")
    print("This is the tradeoff: fixed memory = predictable cost, but older context i")
    

RunnableWithMessageHistory
        │
        │ history_messages_key="history"
        ↓
    key = "history"
        │
        ↓
MessagesPlaceholder("history")
        │
        ↓
Put the messages here

In [ ]:

def demo_summary_memory():
    """Implement conversation summarization."""
    print("=" * 60)
    print("SUMMARY MEMORY")
    print("Summarize older messages to save tokens") 
    print("=" * 60)

    summary_llm=ChatOpenAI(mode="gpt-4o-mini",temperature=0)
    chat_llm=ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

    # A converstaion prompt: summary of old context and recent messages.

    chat_prompt = ChatPromptTemplate.from_messages ([
    ("system", "You are a helpful assistant. Be concise \n\n"
    "Summary of earlier converstion: \n{summary}",
    ),
    MessagesPlaceholder (variable_name="recent_messages"),
    ("human", "{inpu}"),
    ])

    chat_chain= chat_prompt | chat_llm | StrOutputParser()


    # The summarization prompt: compress messages into a running summary
    summarize_prompt = ChatPromptTemplate.from_template(
    "Condense the current summary and new messages into a single updated summary"
    "(2-3 sentences). Preserve all key facts about the user.\n\n"
    "Current summary:\n{current_summary}\n\n"
    "New messages:\n{new_messages}\n\n"
    "Updated summary:"
    )

    summarize_chain=summarize_prompt | summary_llm | StrOutputParser()

    #--STATE--
    running_summary="" #starts empty
    recent_messages=[] #full message objects
    MAX_RECENT=4 #keep last 4 messages (2 exchanges) before summarizing


    exchanges = [
    "My name is Anushka and I'm from India",
    "I work as an AI engineer building RAG systems",
    "I love playing volleyball, going for a swim and a run",
    "I'm learning a LangChain course from Udemy by Paulo",
    "What do you know about me? List everything.",
    ]

    print(f"\nConfig:Keep last{MAX_RECENT} messages, summarize the rest")

    for user_input in exchanges:

        print(f"User: {user_input}")
        # 1. Call the LLM with summary + recent messages + new message
        response=chat_chain.invoke(
            {
                "summary":(running_summary if running_summary else "No prior Conversation"),
                "recent_messages": recent_messages,
                "input": user_input,
            }
        )
        print(f"AI:{response}")
        # 2. Add this exchange to recent messages
        recent_messages.append(HumanMessage(content=user_input))
        recent_messages.append(AIMessage(content=response))

        #3. If recent messages exceed limit, summarize the oldest ones
        if len(recent_messages) > MAX_RECENT:
            #Take the oldest messages that will be summarized away
            messages_to_summarize = recent_messages [:-MAX_RECENT]
            formatted = "\n".join(
            f"{'Human' if isinstance(m, HumanMessage) else 'AI'}: {m.content}"
            for m in messages_to_summarize
            )
        #4. update the running summary
        running_summary=summarize_chain.invoke(
            {
                "current_summary":(running_summary if running_summary else "None yet"),
                "new_messages":formatted,
            }
        )
        #5. Keep only the most recent messages
        recent_messages = recent_messages [-MAX_RECENT:]

        print(
        f" >>> Summarized! Compressed {len (messages_to_summarize)} old messages"
        )
        print(f" >>> Summary: {running_summary}")
        print(f" >>> Recent buffer: {len(recent_messages)} messages")

    print()

    #Final state
    print("=" * 60)
    print("FINAL MEMORY STATE")
    print("=" * 60)
    print(f"\nRunning summary (compressed old context): \n {running_summary}")
    print(f"\nRecent messages kept verbatim ({len (recent_messages)}):")
    for msg in recent_messages:
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f" {role}: {msg.content[:80]}")
    print("\nKey insight: ALL facts preserved (name, city, hobby, course, tutor)")
    print("But token cost stays bounded - old messages are compressed, not deleted")




